# 🧬 Introduction to PyTorch for Genomic Foundation Models

This notebook provides a hand's on introduction to PyTorch for genomic foundation models. The notebook contains solutions to exercises and hidden solutions that can be revealed when ready.

To reveal a solution, click the small **▶ arrow next to “Solution”** in that section.



## 🔧 Setup — Installing Required Libraries

If you are running this in **Google Colab**, the following cell ensures PyTorch and Transformers are installed.

If everything is already installed, you may safely skip this section.


In [1]:
# Uncomment if needed
# CPU only
# !pip install "torch>=2.2" torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# HuggingFace
# !pip install transformers>=4.47.1

## 📦 Imports and Device Check

PyTorch uses a **device abstraction** so the same code runs on CPU or GPU.

If a GPU is available, we will use it automatically.


In [3]:
# hide all warnings
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

try:
    from transformers import AutoTokenizer, AutoModel
    HAS_TRANSFORMERS = True
except:
    HAS_TRANSFORMERS = False

print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PyTorch version: 2.9.1+cu128
Using device: cuda


# 🧱 Section 1 — Tensors

Tensors are the **core building block** in PyTorch.

They behave like NumPy arrays but support:

✔ GPU execution  
✔ Automatic differentiation  
✔ Deep learning operations  

---


In [5]:
x = torch.tensor([1.0, 2.0, 3.0])
x_explicit = x.unsqueeze(1) 
M = torch.randn(2, 3)

print(f"x = {x}")
print(f"M = {M}")
print(f"x.shape = {x.shape}")
print(f"x_explicit.shape = {x_explicit.shape}")
print(f"M.shape = {M.shape}")

# x is treated as a row matching M's columns
print("Matrix multiply M @ x =", M @ x)

x = tensor([1., 2., 3.])
M = tensor([[-1.0154, -0.5349,  0.3707],
        [ 2.1274,  0.1270, -0.7726]])
x.shape = torch.Size([3])
x_explicit.shape = torch.Size([3, 1])
M.shape = torch.Size([2, 3])
Matrix multiply M @ x = tensor([-0.9730,  0.0636])


### 🎯 Exercise 1 — Try Changing the Tensor

Modify the tensor `x` and re-run the cell. Observe what changes.

---

<details>
<summary><strong>Solution</strong></summary>

Change the numbers in `x`. The result of matrix multiplication changes proportionally.

</details>


# 🧠 Section 2 — Automatic Differentiation

PyTorch keeps track of operations to compute **gradients automatically**.

This powers neural network training.

<details>
<summary><strong>Manual Differentiation Step-by-Step</strong></summary>

**Function:**  
$y = 3w^2 + 4w + 1$

**Variable Value:**  
$w = 2.0$

#### 1. Apply the Differentiation Rules
We calculate the derivative $\frac{dy}{dw}$ by differentiating each term separately using the **Power Rule** ($\frac{d}{dx}x^n = nx^{n-1}$):

*   **Differentiating $3w^2$:**  
    Multiply the coefficient by the exponent: $3 \times 2 = 6$.  
    Reduce the exponent by one: $w^{(2-1)} = w^1$.  
    **Result: $6w$**

*   **Differentiating $4w$:**  
    Since $w$ is $w^1$, the exponent becomes $0$.  
    **Result: $4$**

*   **Differentiating $1$:**  
    The derivative of a constant is always zero.  
    **Result: $0$**

#### 2. The Gradient Formula
Combining these results gives us the gradient function:
$$\frac{dy}{dw} = 6w + 4$$

#### 3. Evaluate at $w = 2.0$
Now, substitute the value of $w$ provided in the code:
$$\frac{dy}{dw} = 6(2.0) + 4$$
$$\frac{dy}{dw} = 12.0 + 4$$
$$\frac{dy}{dw} = 16.0$$

</details>

In [7]:
w = torch.tensor(2.0, requires_grad=True)
y = 3*w**2 + 4*w + 1
y.backward()
print("Gradient dy/dw =", w.grad)

Gradient dy/dw = tensor(16.)


# 🏗 Section 3 — Building Your First Model

We now build a **linear regression model** that learns:

$y = 2x + 1$


In [11]:
# Define a simple Linear Regression class inheriting from the base Module
class TinyLinearModel(nn.Module):
    def __init__(self):
        # Initialize the parent class (nn.Module)
        super().__init__()
        # Define a single layer with 1 input and 1 output, including an intercept (bias)
        self.linear = nn.Linear(in_features=1, out_features=1, bias=True)

    # Define the computation graph for the input data
    def forward(self, x):
        return self.linear(x)

# Instantiate the model and move its parameters to the specified hardware (CPU/GPU)
model = TinyLinearModel().to(device)

# Display the model architecture details
model

TinyLinearModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)

## 📊 Section 4 — Create Synthetic Training Data

We generate noisy data so the model can learn.


In [12]:
# Ensure reproducibility by fixing the random number generator seed
torch.manual_seed(42)

# Create 100 points between -1 and 1, reshaping to (100, 1) for model compatibility
x = torch.linspace(-1, 1, 100).unsqueeze(1).to(device)

# Generate targets using a linear function (y = 2x + 1) plus random Gaussian noise
y = 2*x + 1 + 0.1*torch.randn_like(x) 

# Wrap tensors into a Dataset object for easier indexing
dataset = TensorDataset(x, y)

# Create an iterable that yields randomized batches of 16 samples for training
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Log dataset statistics to verify shapes and data types
print(f"Total samples: {len(dataset)}")
for i, t in enumerate(dataset.tensors):
    print(f"Tensor {i} shape: {t.shape}") # Expect [100, 1]
    print(f"Tensor {i} dtype: {t.dtype}") # Expect torch.float32


Total samples: 100
Tensor 0 shape: torch.Size([100, 1])
Tensor 0 dtype: torch.float32
Tensor 1 shape: torch.Size([100, 1])
Tensor 1 dtype: torch.float32


## ⚙ Section 5 — Loss and Optimizer

1. The **loss measures error**.
2. The **optimizer updates parameters**.

In [16]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

Check the details about loss function and optimizer below

<details>
<summary><strong>Core Training Components: Criterion and Optimizer</strong></summary>
In PyTorch, the learning process is governed by two main objects: the **Criterion** (which measures error) and the **Optimizer** (which fixes the error).

---

### 1. The Loss Function: `nn.MSELoss()`
This is the mathematical "yardstick" used to evaluate how well your model is performing.

*   **Full Name:** Mean Squared Error (MSE).
*   **Purpose:** Used for **Regression** tasks (predicting continuous numbers like price, temperature, or a trend).
*   **How it Works:** It calculates the average of the squared differences between the predicted values ($\hat{y}$) and the actual targets ($y$).
*   **The Formula:** 
    $$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
*   **Key Concept:** Because the error is **squared**, large mistakes are penalized much more heavily than small ones. This forces the model to work harder to correct its biggest outliers.

---

### 2. The Optimizer: `torch.optim.SGD`
This is the mathematical "engine" that updates your model's weights based on the errors found by the Loss Function.

*   **Full Name:** Stochastic Gradient Descent (SGD).
*   **Parameters:**
    *   `model.parameters()`: This tells the optimizer which parts of the model (weights and biases) it has permission to change.
    *   `lr=0.1` (Learning Rate): This is the **step size**. It determines how drastically the weights are adjusted after each calculation.
*   **The Learning Process:**
    1.  **Look:** It examines the "gradient" (the direction of the error).
    2.  **Step:** It moves the weights a tiny bit in the opposite direction of the error.
    3.  **Update:** $Weight_{new} = Weight_{old} - (LearningRate \times Gradient)$

---
</details>


# 🚀 Section 6 — Training Loop

This is the core pattern used everywhere, including Hugging Face.


<details>
<summary><strong>Step-by-Step Training Loop Breakdown</strong></summary>

The training loop is the process of repeatedly showing data to the neural network and adjusting its parameters to minimize error.

#### 1. The Epoch Loop
`for epoch in range(50):`
An **Epoch** is one full cycle through the entire dataset. We repeat this process multiple times so the model can iteratively improve its "guess" through practice.

#### 2. The Mini-Batch Loop
`for xb, yb in dataloader:`
Processing 10,000 images at once is memory-intensive. Instead, we split the data into **Mini-Batches** (small groups). 
*   `xb`: The input features for this specific batch.
*   `yb`: The correct target labels for this specific batch.

#### 3. The Forward Pass
`preds = model(xb)`
The model processes the input data (`xb`) through its layers to produce **predictions** (`preds`). 

#### 4. The Loss Calculation
`loss = criterion(preds, yb)`
We use our "yardstick" (MSELoss) to calculate a single number representing how far the model's predictions were from the actual truth.

#### 5. The Optimization Steps (The "Learning")
*   `optimizer.zero_grad()`: **Reset.** Before calculating new gradients, we must clear out the math from the previous batch so it doesn't accumulate.
*   `loss.backward()`: **Backpropagate.** PyTorch travels backward through the model's math to find out which specific weights caused the error.
*   `optimizer.step()`: **Update.** The optimizer nudges the weights slightly in the direction that will make the loss smaller next time.

#### 6. Accumulating Progress
`loss_sum += loss.item() * xb.size(0)`
Because `loss` is the average for a single batch, we multiply it by the number of samples in that batch (`xb.size(0)`) to track the total error across the entire epoch.

#### 7. Monitoring
`print(f"Epoch {epoch+1}: loss={loss_sum/len(dataset):.4f}")`
Every 10 epochs, we print the **Average Loss** (Total Loss / Total Samples). 
*   **A decreasing loss** means the model is successfully learning.
*   **A stagnant loss** means the model has finished training or needs a different learning rate.
  
</details>


In [17]:
# Iterate through the entire dataset 50 times
for epoch in range(50):
    # Initialize a tracker for the cumulative loss of the current epoch
    loss_sum = 0
    
    # Iterate through randomized mini-batches of inputs (xb) and targets (yb)
    for xb, yb in dataloader:
        # Perform the forward pass: compute predictions using the model
        preds = model(xb)
        
        # Calculate the error between predictions and actual targets
        loss = criterion(preds, yb)
        
        # Clear gradients from the previous step to prevent accumulation
        # take many small, frequent steps toward the minimum
        optimizer.zero_grad()
        
        # Compute gradients of the loss with respect to model parameters
        loss.backward()
        
        # Update model weights based on the computed gradients
        optimizer.step()
        
        # Accumulate the total loss (scaling mean loss by batch size for accuracy)
        loss_sum += loss.item() * xb.size(0)

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        # Calculate and display the average loss for the entire dataset
        print(f"Epoch {epoch+1}: loss={loss_sum/len(dataset):.4f}")


Epoch 10: loss=0.0088
Epoch 20: loss=0.0087
Epoch 30: loss=0.0088
Epoch 40: loss=0.0087
Epoch 50: loss=0.0087


## 🔍 Section 7 — Inspect Learned Parameters

In [18]:
w = model.linear.weight.item()
b = model.linear.bias.item()
print(w, b)

1.9934093952178955 1.0021049976348877


# 🧪 Section 8 — Optional Interactive Classification Demo

We train a tiny classifier.


In [21]:
from sklearn.datasets import make_moons
import torch.nn.functional as F

X, y = make_moons(n_samples=500, noise=0.2, random_state=0)
X = torch.tensor(X, dtype=torch.float32).to(device)
y = torch.tensor(y, dtype=torch.long).to(device)

loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )
    def forward(self, x):
        return self.net(x)

clf = TinyClassifier().to(device)
opt = torch.optim.Adam(clf.parameters(), lr=1e-2)

for epoch in range(20):
    correct = 0
    total = 0
    for xb, yb in loader:
        logits = clf(xb)
        loss = F.cross_entropy(logits, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        correct += (logits.argmax(1)==yb).sum().item()
        total += xb.size(0)
    if (epoch+1)%5==0:
        print(f"Epoch {epoch+1}: accuracy={correct/total:.3f}")

ImportError: cannot import name '_spbase' from 'scipy.sparse._base' (/biocorelab/BIX/util/miniconda3/envs/gfm-workshop/lib/python3.11/site-packages/scipy/sparse/_base.py)

# 🔗 Section 9 — Connecting to Hugging Face Genomic LLMs

This demonstrates how pretrained models still use PyTorch tensors under the hood.


In [32]:
model_name = "RaphaelMourad/Mistral-DNA-v1-17M-hg38"

if HAS_TRANSFORMERS:
    print(f"--- Loading Model: {model_name} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model_hf = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)
    print(f"Model loaded successfully on device: {device}\n")

    # 1. Prepare Input
    seq = "ACGT" * 20
    print(f"Input DNA Sequence (Length {len(seq)}): {seq[:20]}...")
    
    # 2. Tokenization
    inputs = tokenizer(seq, return_tensors="pt").to(device)
    print(f"Tokenized IDs (Input IDs): {inputs['input_ids']}")
    print(f"Number of tokens created: {inputs['input_ids'].shape[1]}\n")

    # 3. Inference
    print("--- Running Forward Pass ---")
    with torch.no_grad():
        outputs = model_hf(**inputs)

    # 4. Explain Output Shape
    shape = outputs.last_hidden_state.shape
    print(f"Output Shape: {shape}")
    print(f"Interpretation: {shape[0]} Sequence | {shape[1]} Tokens | {shape[2]} Hidden Dimensions (Embedding Size)")
    
    # Accessing the vector for the first token
    first_token_vector = outputs.last_hidden_state[0, 0, :5]
    print(f"First 5 values of the first token's embedding: {first_token_vector}")

else:
    print("Error: 'transformers' library not installed. Please run !pip install transformers")


--- Loading Model: RaphaelMourad/Mistral-DNA-v1-17M-hg38 ---
Model loaded successfully on device: cuda

Input DNA Sequence (Length 80): ACGTACGTACGTACGTACGT...
Tokenized IDs (Input IDs): tensor([[   1,    5,  194,  194,  194,  194,  194,  194,  194,  194,  194,  194,
          194,  194,  194,  194,  194,  194,  194,  194,  194,    6, 1049,    2]],
       device='cuda:0')
Number of tokens created: 24

--- Running Forward Pass ---
Output Shape: torch.Size([1, 24, 256])
Interpretation: 1 Sequence | 24 Tokens | 256 Hidden Dimensions (Embedding Size)
First 5 values of the first token's embedding: tensor([-0.0653, -0.2077,  0.2044, -0.4206,  0.1455], device='cuda:0')


# 🎯 Final Summary

You now understand:

✔ Tensors  
✔ Autograd  
✔ Model building  
✔ Training loops  
✔ Hugging Face integration  

These are the foundations of **genomic foundation model fine‑tuning**.
